Feature Engineering Workflow

1. Feature Engineering Objectives & Scope
2. Feature Design Rationale (EDA-Driven)
3. Transaction-Level Feature Creation
4. Customer-Level Aggregated Features
5. Behavioral Segmentation Features
6. Promotion & Engagement Features
7. Feature Validation & Sanity Checks
8. Final Feature Dataset Output

---

# **Feature Engineering Objectives & Scope** 

---

The objective of this notebook is to derive meaningful, interpretable, and business-aligned features that capture customer spending behavior, engagement intensity, and transactional patterns.

All features are derived exclusively from cleaned transactional data and are designed to support downstream exploratory analysis, SQL-based reporting, and business intelligence dashboards.


In [211]:
#Dependencies

import pandas as pd

In [212]:
df = pd.read_csv('../data/processed/consumer_data_cleaned.csv')
print("Data Size: ", df.shape)
df.head(10)

Data Size:  (3900, 19)


,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,promo_code_used,previous_purchases,payment_method,frequency_of_purchases,purchase_outlier_flag
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,True,True,14,Venmo,Fortnightly,False
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,True,True,2,Cash,Fortnightly,False
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,True,True,23,Credit Card,Weekly,False
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,True,True,49,Paypal,Weekly,False
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,True,True,31,Paypal,Annually,False
5,6,46,Male,Sneakers,Footwear,20,Wyoming,M,White,Summer,2.9,Yes,Standard,True,True,14,Venmo,Weekly,False
6,7,63,Male,Shirt,Clothing,85,Montana,M,Gray,Fall,3.2,Yes,Free Shipping,True,True,49,Cash,Quarterly,False
7,8,27,Male,Shorts,Clothing,34,Louisiana,L,Charcoal,Winter,3.2,Yes,Free Shipping,True,True,19,Credit Card,Weekly,False
8,9,26,Male,Coat,Outerwear,97,West Virginia,L,Silver,Summer,2.6,Yes,Express,True,True,8,Venmo,Annually,False
9,10,57,Male,Handbag,Accessories,31,Missouri,M,Pink,Spring,4.8,Yes,2-Day Shipping,True,True,4,Cash,Quarterly,False


---

# **Feature Design Rationale (EDA-Driven)** 

---

Feature engineering decisions are guided by exploratory findings, including observed variability in purchase amount, purchase frequency, promotional usage, and customer engagement behavior. The engineered features aim to consolidate raw transactional data into higher-level behavioral signals.

---

# **Transaction-Level Feature Creation** 

---

#### **1. Feature Consolidation & Redundancy Reduction** 

In [213]:
df[['discount_applied', 'promo_code_used']].head()

,discount_applied,promo_code_used
0,True,True
1,True,True
2,True,True
3,True,True
4,True,True


In [214]:
(df['discount_applied'] == df['promo_code_used']).all()

np.True_

In [215]:
#Since both columns are identical, we can drop one of them
df.drop('promo_code_used', axis=1, inplace=True)
df.head()

,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,previous_purchases,payment_method,frequency_of_purchases,purchase_outlier_flag
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,True,14,Venmo,Fortnightly,False
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,True,2,Cash,Fortnightly,False
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,True,23,Credit Card,Weekly,False
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,True,49,Paypal,Weekly,False
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,True,31,Paypal,Annually,False


#### **2. Customer Age Grouping** 

In [216]:
df["age_group"] = pd.qcut( df["age"], q=4, labels=["Teen", "Young Adult", "Adult", "Senior"] )

df.head()


,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,previous_purchases,payment_method,frequency_of_purchases,purchase_outlier_flag,age_group
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,True,14,Venmo,Fortnightly,False,Adult
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,True,2,Cash,Fortnightly,False,Teen
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,True,23,Credit Card,Weekly,False,Adult
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,True,49,Paypal,Weekly,False,Teen
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,True,31,Paypal,Annually,False,Adult


#### **3. High-value Transaction Creation** 

In [217]:
median_purchase = df["purchase_amount"].median()

print(median_purchase)

df["high_value_transaction_flag"] = (
    df["purchase_amount"] > median_purchase
).astype(int)

df

60.0


,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,previous_purchases,payment_method,frequency_of_purchases,purchase_outlier_flag,age_group,high_value_transaction_flag
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,True,14,Venmo,Fortnightly,False,Adult,0
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,True,2,Cash,Fortnightly,False,Teen,1
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,True,23,Credit Card,Weekly,False,Adult,1
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,True,49,Paypal,Weekly,False,Teen,1
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,True,31,Paypal,Annually,False,Adult,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3895,3896,40,Female,Hoodie,Clothing,28,Virginia,L,Turquoise,Summer,4.2,No,2-Day Shipping,False,32,Venmo,Weekly,False,Young Adult,0
3896,3897,52,Female,Backpack,Accessories,49,Iowa,L,White,Spring,4.5,No,Store Pickup,False,41,Bank Transfer,Bi-Weekly,False,Adult,0
3897,3898,46,Female,Belt,Accessories,33,New Jersey,L,Green,Spring,2.9,No,Standard,False,24,Venmo,Quarterly,False,Adult,0
3898,3899,44,Female,Shoes,Footwear,77,Minnesota,S,Brown,Summer,3.8,No,Express,False,24,Venmo,Weekly,False,Young Adult,1


#### **4. Promotion Exposure Flag** 

In [218]:
df["promotion_exposure_flag"] = (
    (df["discount_applied"] == True) #Boolean type
).astype(int)

df.head()


,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,...,subscription_status,shipping_type,discount_applied,previous_purchases,payment_method,frequency_of_purchases,purchase_outlier_flag,age_group,high_value_transaction_flag,promotion_exposure_flag
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,...,Yes,Express,True,14,Venmo,Fortnightly,False,Adult,0,1
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,...,Yes,Express,True,2,Cash,Fortnightly,False,Teen,1,1
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,...,Yes,Free Shipping,True,23,Credit Card,Weekly,False,Adult,1,1
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,...,Yes,Next Day Air,True,49,Paypal,Weekly,False,Teen,1,1
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,...,Yes,Free Shipping,True,31,Paypal,Annually,False,Adult,0,1


#### **5. Convenience Preference Flag** 

In [219]:
df["express_shipping_flag"] = (
    df["shipping_type"].isin(["Express", "Next Day Air"])
).astype(int)

df.head()


,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,...,shipping_type,discount_applied,previous_purchases,payment_method,frequency_of_purchases,purchase_outlier_flag,age_group,high_value_transaction_flag,promotion_exposure_flag,express_shipping_flag
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,...,Express,True,14,Venmo,Fortnightly,False,Adult,0,1,1
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,...,Express,True,2,Cash,Fortnightly,False,Teen,1,1,1
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,...,Free Shipping,True,23,Credit Card,Weekly,False,Adult,1,1,0
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,...,Next Day Air,True,49,Paypal,Weekly,False,Teen,1,1,1
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,...,Free Shipping,True,31,Paypal,Annually,False,Adult,0,1,0


#### **6. Transformation of Purchase Frequency into Numerical Scale**

In [220]:
df['frequency_of_purchases'].unique()

array(['Fortnightly', 'Weekly', 'Annually', 'Quarterly', 'Bi-Weekly',
       'Monthly', 'Every 3 Months'], dtype=object)

In [221]:
purchase_frequenc_mapping = {
    'Fortnightly': 14,
    'Weekly': 7,
    'Monthly': 30,
    'Quarterly': 90,
    'Bi-Weekly': 14,
    'Annually': 365,
    'Every 3 Months': 90
}

df['purchase_frequency_days'] = df['frequency_of_purchases'].map(purchase_frequenc_mapping)
df.head()

,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,...,discount_applied,previous_purchases,payment_method,frequency_of_purchases,purchase_outlier_flag,age_group,high_value_transaction_flag,promotion_exposure_flag,express_shipping_flag,purchase_frequency_days
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,...,True,14,Venmo,Fortnightly,False,Adult,0,1,1,14
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,...,True,2,Cash,Fortnightly,False,Teen,1,1,1,14
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,...,True,23,Credit Card,Weekly,False,Adult,1,1,0,7
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,...,True,49,Paypal,Weekly,False,Teen,1,1,1,7
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,...,True,31,Paypal,Annually,False,Adult,0,1,0,365


In [222]:
df['purchase_frequency_days'].isna().sum() #Validation that all columns are filled


np.int64(0)

#### **Important Note**

- All rows are unique representing unique customer base. Dataset choosen here gives customers' current transactions and count of their previous purchases whose details are not part of this dataset.

In [223]:
df.shape

(3900, 23)

In [224]:
df["customer_id"].unique().shape

(3900,)

---

# **Customer-Level Aggregated Features** 

---
Transaction Level Data -> Customer-Level Behaviour

This section does not make sense for this EDA, because all the records are unique customer ids. So, Customer Level aggregation does not make sense here.

However keeping the code commented for consistency.

#### **Aggregation Logic** 

In [225]:
# customer_features = (
#     df.groupby("customer_id")
#     .agg(
#         age=("age", "max"), 
#         total_spend=("purchase_amount", "sum"),
#         avg_spend=("purchase_amount", "mean"),
#         max_spend=("purchase_amount", "max"),
#         purchase_count=("purchase_amount", "count"),
#         high_value_txn_count=("high_value_transaction_flag", "sum"), 
#         promotion_txn_count=("promotion_exposure_flag", "sum"), 
#         avg_review_rating=("review_rating", "mean"),
#         express_shipping_count=("express_shipping_flag", "sum"),
#         max_previous_purchases=("previous_purchases", "max")
#     )
#     .reset_index()
# )

# customer_features.head()


Insights enabled:

- Customer lifetime value proxy
- Loyalty strength
- Promotion sensitivity
- Satisfaction trend

---

# **Behavioral Segmentation Features** 

---

This section also not required in this EDA. Since customer level data is only unique rows.

#### **1. Spend Tier** 

In [226]:
# customer_features["spend_tier"] = pd.qcut(
#     customer_features["total_spend"],
#     q=3,
#     labels=["Low Spender", "Mid Spender", "High Spender"]
# )

# customer_features.head()


#### **2. Engagement Level** 

In [227]:
# customer_features.head()

#Engagement Levele not requires here -> since purchase_count is 1 for all customers

---

# **Promotion & Engagement Metrics** 

---

This section also not required in this EDA. Since customer level data is only unique rows.

#### **1. Promotion Dependency Ratio** 

In [228]:
# customer_features["promotion_dependency_ratio"] = (
#     customer_features["promotion_txn_count"] /
#     customer_features["purchase_count"]
# ).round(2)

# customer_features.head()

- High ratio → discount-driven customers
- Low ratio → organic loyalty

#### **2. High Value Transation Ratio**

In [229]:
# customer_features["high_value_ratio"] = (
#     customer_features["high_value_txn_count"] /
#     customer_features["purchase_count"]
# ).round(2)

# customer_features.head()

---

# **Feature Validation and Sanity Check** 

---

In [230]:
df.isna().sum()

customer_id                    0
age                            0
gender                         0
item_purchased                 0
category                       0
purchase_amount                0
location                       0
size                           0
color                          0
season                         0
review_rating                  0
subscription_status            0
shipping_type                  0
discount_applied               0
previous_purchases             0
payment_method                 0
frequency_of_purchases         0
purchase_outlier_flag          0
age_group                      0
high_value_transaction_flag    0
promotion_exposure_flag        0
express_shipping_flag          0
purchase_frequency_days        0
dtype: int64

In [231]:
df.describe()

,customer_id,age,purchase_amount,review_rating,previous_purchases,high_value_transaction_flag,promotion_exposure_flag,express_shipping_flag,purchase_frequency_days
count,3900.000000,3900.000000,3900.000000,3900.000000,3900.000000,3900.000000,3900.000000,3900.000000,3900.000000
mean,1950.500000,44.068462,59.764359,3.750051,25.351538,0.490769,0.430000,0.331795,89.133077
std,1125.977353,15.207589,23.685392,0.713590,14.447125,0.499979,0.495139,0.470918,119.037566
min,1.000000,18.000000,20.000000,2.500000,1.000000,0.000000,0.000000,0.000000,7.000000
25%,975.750000,31.000000,39.000000,3.100000,13.000000,0.000000,0.000000,0.000000,14.000000
50%,1950.500000,44.000000,60.000000,3.800000,25.000000,0.000000,0.000000,0.000000,30.000000
75%,2925.250000,57.000000,81.000000,4.400000,38.000000,1.000000,1.000000,1.000000,90.000000
max,3900.000000,70.000000,100.000000,5.000000,50.000000,1.000000,1.000000,1.000000,365.000000


In [232]:
df.head()

,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,...,discount_applied,previous_purchases,payment_method,frequency_of_purchases,purchase_outlier_flag,age_group,high_value_transaction_flag,promotion_exposure_flag,express_shipping_flag,purchase_frequency_days
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,...,True,14,Venmo,Fortnightly,False,Adult,0,1,1,14
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,...,True,2,Cash,Fortnightly,False,Teen,1,1,1,14
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,...,True,23,Credit Card,Weekly,False,Adult,1,1,0,7
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,...,True,49,Paypal,Weekly,False,Teen,1,1,1,7
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,...,True,31,Paypal,Annually,False,Adult,0,1,0,365


---

# **Final Feature Dataset Output**

---

In [233]:
df.to_csv("../data/final/consumer_final_data.csv", index=False)


Finalized dataset csv saved with some new relevant features introduced driven by business thinking.

---

# **Next Steps & Assumptions**

---

Subsequent steps will focus on:

- Post Feature Validation
- SQL Business Queries
- Dashboarding in PowerBI